logistic regression over ad text to score each ad's probability of being election-content. positives are advocacy ads from ±4 weeks around election day, negatives are advocacy ads from the far edges of the window. structural labels, no advertiser identity in either class. then score every v3 advocacy ad and look for advertisers whose election-content score spikes around election day even though their overall spend pattern says ongoing.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, lower, regexp_replace, concat_ws, array_join,
    sum as spark_sum, count as spark_count, date_trunc, desc, abs as spark_abs,
)
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover, CountVectorizer, IDF,
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

spark = SparkSession.builder \
    .appName('FB_API_election_content_clf') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

In [ ]:
V3_PATH = '/user/s3348393/main/preprocessing/v3/parquet'
ELECTION_DATE = pd.Timestamp('2022-05-21')
HOT_ZONE_WEEKS  = 4   # ±4 weeks around election day defines positive class
EDGE_ZONE_WEEKS = 8   # first / last 8 weeks of the window define negative class
WINDOW_START = pd.Timestamp('2021-11-21')
WINDOW_END   = pd.Timestamp('2022-11-21')

load v3, restrict to advocacy. one row per ad.

In [ ]:
v3 = spark.read.parquet(V3_PATH).filter(col('match_type').isNull())

# combine body + title arrays into one text blob per ad
v3 = v3.withColumn('text', concat_ws(' ',
    array_join('creative_bodies', ' '),
    array_join('creative_link_titles', ' ')))
v3 = v3.withColumn('text', lower(regexp_replace('text', r'[^a-z\s]', ' ')))

v3 = v3.filter(col('text').isNotNull() & (col('text') != ''))
v3 = v3.cache()
print(f'Advocacy ads with text: {v3.count():,}')

build the labelled training set. positives are ads from ±4 weeks around election day, negatives are ads from the first 8 and last 8 weeks of the analysis window. labels are purely structural (defined by ad date), so the model can only learn content patterns, not advertiser identity.

In [ ]:
hot_start  = (ELECTION_DATE - pd.Timedelta(weeks=HOT_ZONE_WEEKS)).date().isoformat()
hot_end    = (ELECTION_DATE + pd.Timedelta(weeks=HOT_ZONE_WEEKS)).date().isoformat()
edge_pre   = (WINDOW_START + pd.Timedelta(weeks=EDGE_ZONE_WEEKS)).date().isoformat()
edge_post  = (WINDOW_END   - pd.Timedelta(weeks=EDGE_ZONE_WEEKS)).date().isoformat()

print(f'positive (hot zone):   {hot_start} to {hot_end}')
print(f'negative (pre-edge):   < {edge_pre}')
print(f'negative (post-edge):  > {edge_post}')

labelled = v3.withColumn('label',
    when((col('ad_creation_date') >= lit(hot_start)) &
         (col('ad_creation_date') <= lit(hot_end)), lit(1.0))
    .when((col('ad_creation_date') < lit(edge_pre)) |
          (col('ad_creation_date') > lit(edge_post)), lit(0.0))
    .otherwise(lit(None))
)

train_full = labelled.filter(col('label').isNotNull()).cache()

print('\nClass balance:')
train_full.groupBy('label').count().show()

tokenise, drop stopwords, vectorise. uses spark mllib's pipeline so the same transform applies at inference time.

In [ ]:
tokenizer = RegexTokenizer(inputCol='text', outputCol='tokens',
                            pattern=r'\s+', minTokenLength=3)

stop_words = list(set(StopWordsRemover.loadDefaultStopWords('english')) | {
    'australia', 'australian', 'australians', 'sign', 'petition', 'please',
    'today', 'help', 'one', 'us', 'go', 'http', 'https', 'www', 'com', 'au',
})

remover = StopWordsRemover(inputCol='tokens', outputCol='tokens_clean',
                            stopWords=stop_words)

cv = CountVectorizer(inputCol='tokens_clean', outputCol='tf',
                      minDF=20, maxDF=0.3, vocabSize=5000)
idf = IDF(inputCol='tf', outputCol='features')
lr  = LogisticRegression(featuresCol='features', labelCol='label',
                          maxIter=50, regParam=0.01)

pipeline = Pipeline(stages=[tokenizer, remover, cv, idf, lr])

train/test split 80/20, fit, evaluate.

In [ ]:
train, test = train_full.randomSplit([0.8, 0.2], seed=42)
train = train.cache()
test = test.cache()

print(f'train: {train.count():,}')
print(f'test:  {test.count():,}')

model = pipeline.fit(train)
pred = model.transform(test).cache()

auc and confusion-matrix metrics on the held-out test set.

In [ ]:
auc_eval = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
auc = auc_eval.evaluate(pred)
print(f'AUC: {auc:.4f}')

for metric in ['accuracy', 'weightedPrecision', 'weightedRecall', 'f1']:
    e = MulticlassClassificationEvaluator(labelCol='label',
                                            predictionCol='prediction',
                                            metricName=metric)
    print(f'{metric}: {e.evaluate(pred):.4f}')

print('\nConfusion matrix (label × prediction):')
pred.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

inspect the top features. if the model is learning election content, top positive-weighted tokens should be election-themed. if it's learning style or noise, they won't be.

In [ ]:
cv_model = model.stages[2]
lr_model = model.stages[4]

vocab = cv_model.vocabulary
coefs = lr_model.coefficients.toArray()

feature_weights = pd.DataFrame({'token': vocab, 'weight': coefs})

print('Top 30 election-content features (positive weight):')
display(feature_weights.sort_values('weight', ascending=False).head(30).reset_index(drop=True))

print('\nTop 30 non-election features (negative weight):')
display(feature_weights.sort_values('weight', ascending=True).head(30).reset_index(drop=True))

apply the model to every v3 advocacy ad, extracting the probability of class 1 (election-content).

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

extract_prob = udf(lambda v: float(v[1]), DoubleType())

scored = (model.transform(v3)
    .withColumn('election_prob', extract_prob('probability'))
    .select('id', 'bylines', 'page_name', 'ad_creation_date',
            'spend_mid', 'text', 'election_prob')
    .cache())

print(f'Scored ads: {scored.count():,}')

false-positive check. take ads with the highest election_prob but from advertisers we'd intuitively expect to score low (i.e. ongoing humanitarian / environmental advertisers, far away from election day). if the model is well-calibrated, these should genuinely contain election-themed content. if not, the model is over-firing.

In [ ]:
# ads from outside the hot zone with very high election_prob
fp_check = (scored
    .filter((col('ad_creation_date') < lit(hot_start)) |
            (col('ad_creation_date') > lit(hot_end)))
    .filter(col('election_prob') > 0.85)
    .orderBy(desc('election_prob'))
    .limit(30)
    .toPandas())

fp_check['text_preview'] = fp_check['text'].str[:120]
print('High-scoring ads from outside hot zone (potential false positives):')
display(fp_check[['bylines', 'ad_creation_date', 'election_prob', 'text_preview']])

false-negative check. ads inside the hot zone but scoring very low. these should be genuinely non-election content (humanitarian campaigns running during election period). if many of them look election-themed, the model is missing signal.

In [ ]:
fn_check = (scored
    .filter((col('ad_creation_date') >= lit(hot_start)) &
            (col('ad_creation_date') <= lit(hot_end)))
    .filter(col('election_prob') < 0.15)
    .orderBy('election_prob')
    .limit(30)
    .toPandas())

fn_check['text_preview'] = fn_check['text'].str[:120]
print('Low-scoring ads from inside hot zone (potential false negatives):')
display(fn_check[['bylines', 'ad_creation_date', 'election_prob', 'text_preview']])

score-distribution sanity check. score histogram and class-conditional distributions.

In [ ]:
score_pdf = scored.select('ad_creation_date', 'election_prob').toPandas()
score_pdf['period'] = pd.cut(
    pd.to_datetime(score_pdf['ad_creation_date']),
    bins=[WINDOW_START, pd.Timestamp(hot_start), pd.Timestamp(hot_end), WINDOW_END],
    labels=['pre-edge', 'hot zone', 'post-edge'],
)

fig, ax = plt.subplots(figsize=(10, 5))
for period, sub in score_pdf.groupby('period', observed=True):
    ax.hist(sub['election_prob'], bins=40, alpha=0.5, label=period, density=True)
ax.set_xlabel('election_prob')
ax.set_ylabel('density')
ax.set_title('score distribution by period')
ax.legend()
plt.tight_layout()
plt.show()

aggregate per advertiser. for each byline compute total spend and spend-weighted mean election_prob. high mean = advertiser's portfolio is dominated by election content. low mean = mostly non-election content.

In [ ]:
from pyspark.sql.functions import sum as spark_sum, avg

per_advertiser = (scored
    .filter(col('bylines').isNotNull() & col('spend_mid').isNotNull())
    .withColumn('weighted_prob', col('election_prob') * col('spend_mid'))
    .groupBy('bylines')
    .agg(
        spark_sum('spend_mid').alias('total_spend'),
        spark_sum('weighted_prob').alias('weighted_prob_sum'),
        spark_count('*').alias('n_ads'),
    )
    .withColumn('mean_election_prob_weighted',
                col('weighted_prob_sum') / col('total_spend'))
    .orderBy(desc('total_spend'))
    .toPandas())

top30 = per_advertiser.head(30).reset_index(drop=True)
display(top30[['bylines', 'total_spend', 'n_ads', 'mean_election_prob_weighted']])

per-advertiser per-week aggregation for the top 20. for each (byline, week) compute the spend-weighted election_prob. plot as a small-multiples chart. a pivot pattern looks like a hump around may 2022 for an advertiser whose spend curve in nb 05 looked flat (ongoing).

In [ ]:
import math

top20_bylines = top30.head(20)['bylines'].tolist()

weekly_scores = (scored
    .filter(col('bylines').isin(top20_bylines) & col('spend_mid').isNotNull())
    .withColumn('week', date_trunc('week', 'ad_creation_date'))
    .withColumn('weighted_prob', col('election_prob') * col('spend_mid'))
    .groupBy('week', 'bylines')
    .agg(
        spark_sum('weighted_prob').alias('weighted_prob_sum'),
        spark_sum('spend_mid').alias('week_spend'),
    )
    .withColumn('week_election_prob', col('weighted_prob_sum') / col('week_spend'))
    .orderBy('bylines', 'week')
    .toPandas())

weekly_scores['week'] = pd.to_datetime(weekly_scores['week'])

ncols = 4
nrows = math.ceil(len(top20_bylines) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3 * nrows),
                          sharex=True, sharey=True, squeeze=False)
axes = axes.flatten()

for i, byline in enumerate(top20_bylines):
    ax = axes[i]
    sub = weekly_scores[weekly_scores['bylines'] == byline].sort_values('week')
    ax.plot(sub['week'], sub['week_election_prob'], color='#1f77b4', linewidth=1.3)
    ax.fill_between(sub['week'], sub['week_election_prob'], color='#1f77b4', alpha=0.3)
    ax.axvline(ELECTION_DATE, linestyle='--', color='red', alpha=0.5)
    ax.set_title(byline[:32], fontsize=9)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.grid(alpha=0.3)

for j in range(len(top20_bylines), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('weekly spend-weighted election_prob, top 20 advertisers',
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

pivot detection. for each advertiser compute the ratio of mean election_prob inside the hot zone vs outside. ratio >> 1 means their content shifted toward election themes around may 2022 even if their overall spend pattern didn't.

In [ ]:
from pyspark.sql.functions import mean as spark_mean

advertiser_pivot = (scored
    .filter(col('bylines').isin(top20_bylines))
    .withColumn('in_hot',
                (col('ad_creation_date') >= lit(hot_start)) &
                (col('ad_creation_date') <= lit(hot_end)))
    .groupBy('bylines', 'in_hot')
    .agg(spark_mean('election_prob').alias('mean_prob'),
         spark_count('*').alias('n_ads'))
    .toPandas())

pivot_wide = advertiser_pivot.pivot_table(
    index='bylines', columns='in_hot', values='mean_prob').reset_index()
pivot_wide.columns = ['bylines', 'mean_prob_outside', 'mean_prob_inside']
pivot_wide['shift'] = (pivot_wide['mean_prob_inside'] -
                       pivot_wide['mean_prob_outside'])
pivot_wide['ratio'] = (pivot_wide['mean_prob_inside'] /
                       pivot_wide['mean_prob_outside'].replace(0, np.nan))
pivot_wide = pivot_wide.sort_values('shift', ascending=False).reset_index(drop=True)

display(pivot_wide)

summary findings. an advertiser landing in `ongoing` from nb 05 (steady dollar flow) but with a large positive shift here is the pivot pattern we were looking for. an advertiser in `election_only` with a high mean_prob_inside is consistent with both methodologies.